# 03 - Baseline vs RAG Q&A + Raw Traces
Run one question in both modes and inspect the raw JSONL telemetry records.

In [1]:
from pathlib import Path
from rag_telemetry_evals.config import get_settings
from rag_telemetry_evals.pipeline import answer_question
settings = get_settings()
question = 'When must emergency rollback happen for API incidents?'

In [2]:
baseline = await answer_question(settings, question=question, use_rag=False)
baseline['answer']

'I do not know.'

In [3]:
rag = await answer_question(settings, question=question, use_rag=True)
{'answer': rag['answer'], 'retrieved_sources': [r['source'] for r in rag['retrieved']]}

{'answer': 'From operations_runbook.md: # Operations Runbook Nimbus Analytics runs a 24x7 primary on-call rotation. The paging system name is **Solar Pager**. Incident severity uses four levels: - **S0**: complete production outage - **S1**: critical degradation for paying users - **S2**: moderate degradation with workaround - **S3**: low-impact issue For API incidents, emergency rollback is mandatory if **p95 latency exceeds 1200 ms for 10 consecutive minutes**. The incident bridge template codename is **Amber-7**. Generated via extractive fallback because the local model request timed out.',
 'retrieved_sources': ['operations_runbook.md',
  'release_protocol.md',
  'support_handbook.md',
  'data_policy.md']}

In [4]:
trace_lines = Path(settings.trace_file).read_text(encoding='utf-8').splitlines()
trace_lines[-5:]

['{"trace_id":"ask-1781263726262","span_name":"generate_baseline","status":"error","start_time_utc":"2026-06-12T11:28:46.262+00:00","end_time_utc":"2026-06-12T11:29:01.279+00:00","latency_ms":15016.487,"attributes":{"model":"phi3.5:3.8b","error":"TimeoutError: "}}',
 '{"trace_id":"ask-1781263726262","span_name":"answer_baseline","status":"ok","start_time_utc":"2026-06-12T11:28:46.262+00:00","end_time_utc":"2026-06-12T11:29:01.281+00:00","latency_ms":15019.075,"attributes":{"question_chars":54}}',
 '{"trace_id":"ask-1781263741349","span_name":"retrieve","status":"ok","start_time_utc":"2026-06-12T11:29:01.349+00:00","end_time_utc":"2026-06-12T11:29:08.116+00:00","latency_ms":6766.696,"attributes":{"question_chars":54,"top_k":4,"retrieval_mode":"vector","n_retrieved":4,"top_score":0.483975}}',
 '{"trace_id":"ask-1781263741349","span_name":"generate_rag","status":"error","start_time_utc":"2026-06-12T11:29:08.116+00:00","end_time_utc":"2026-06-12T11:29:23.133+00:00","latency_ms":15017.081,"